In [ ]:
!pip install pyspark==3.5.0 delta-spark

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("delta-fix") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

## 1. Data Ingestion and Setup

### 1(a) Load CSV into Delta Table

In [ ]:
dataPath = "/content/employee_data.csv"

# Configure Delta Lake Path
deltaPath = "/content/tmp"

from pyspark.sql import SparkSession

# Read CSV
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv(dataPath))

# Save as Delta
df.write.format("delta").mode("overwrite").save(deltaPath)

In [ ]:
df.head(10)

[Row(employee_id=1.0, age=33.0, city='New York', department='Finance', salary=58101.0, join_date=datetime.date(2023, 7, 26), performance_score=3.0),
 Row(employee_id=2.0, age=58.0, city='New York', department='Marketing', salary=123851.0, join_date=datetime.date(2021, 12, 25), performance_score=4.7),
 Row(employee_id=3.0, age=53.0, city='Los Angeles', department='HR', salary=56372.0, join_date=datetime.date(2022, 2, 11), performance_score=4.0),
 Row(employee_id=4.0, age=35.0, city='Phoenix', department='Marketing', salary=None, join_date=datetime.date(2023, 12, 4), performance_score=4.3),
 Row(employee_id=5.0, age=39.0, city='Houston', department='Marketing', salary=89510.0, join_date=datetime.date(2024, 9, 9), performance_score=2.9),
 Row(employee_id=6.0, age=60.0, city='New York', department='Finance', salary=41899.0, join_date=datetime.date(2016, 11, 12), performance_score=3.4),
 Row(employee_id=7.0, age=52.0, city='New York', department='Finance', salary=126963.0, join_date=datetim

### 1(b) Check for Missing and Duplicate Rows

In [ ]:
from pyspark.sql.functions import col, when, count

# Missing value count
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# Duplicate row count
df.count() - df.dropDuplicates().count()

+-----------+---+----+----------+------+---------+-----------------+
|employee_id|age|city|department|salary|join_date|performance_score|
+-----------+---+----+----------+------+---------+-----------------+
|        240|234| 265|       246|   251|      245|              242|
+-----------+---+----+----------+------+---------+-----------------+



0

## 2. Schema Enforcement

### 2(a) Define and Enforce Schema

In [ ]:
# Load the original Delta table
df = spark.read.format("delta").load(deltaPath)
df.printSchema()

root
 |-- employee_id: double (nullable = true)
 |-- age: double (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- join_date: date (nullable = true)
 |-- performance_score: double (nullable = true)



In [ ]:
from pyspark.sql.types import *
import datetime

# Define schema: all numeric types as DoubleType for consistency with Delta table
mismatch_schema = StructType([
    StructField("employee_id", DoubleType(), False),
    StructField("age", DoubleType(), True),
    StructField("city", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("join_date", DateType(), True),
    StructField("performance_score", DoubleType(), True),
    StructField("courses", StringType(), True)
])

# Match schema by using correct data types (explicitly using floats for numeric fields)
items = [
    (9999.0, 28.0, "Chicago", "HR", 60000.0, datetime.date(2023, 9, 1), 3.8, "SEAS8515")
]

# Create DataFrame with schema applied
mismatch_df = spark.createDataFrame(items, schema=mismatch_schema)

# Display schema and contents
mismatch_df.printSchema()
mismatch_df.show()

root
 |-- employee_id: double (nullable = false)
 |-- age: double (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- join_date: date (nullable = true)
 |-- performance_score: double (nullable = true)
 |-- courses: string (nullable = true)

+-----------+----+-------+----------+-------+----------+-----------------+--------+
|employee_id| age|   city|department| salary| join_date|performance_score| courses|
+-----------+----+-------+----------+-------+----------+-----------------+--------+
|     9999.0|28.0|Chicago|        HR|60000.0|2023-09-01|              3.8|SEAS8515|
+-----------+----+-------+----------+-------+----------+-----------------+--------+



### 2(b) Mismatched Schema

In [ ]:
mismatch_df.write.format("delta").mode("append").save(deltaPath)

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: f8e4da23-2c06-4360-bf67-e19aab13c4f9).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- employee_id: double (nullable = true)
-- age: double (nullable = true)
-- city: string (nullable = true)
-- department: string (nullable = true)
-- salary: double (nullable = true)
-- join_date: date (nullable = true)
-- performance_score: double (nullable = true)


Data schema:
root
-- employee_id: double (nullable = true)
-- age: double (nullable = true)
-- city: string (nullable = true)
-- department: string (nullable = true)
-- salary: double (nullable = true)
-- join_date: date (nullable = true)
-- performance_score: d

## 3. Data Updates

### 3(a) Append Data

In [ ]:
(mismatch_df.write.format("delta").mode("append")
  .option("mergeSchema", "true")
  .save(deltaPath))

In [ ]:
spark.read.format("delta").load(deltaPath).filter("employee_id = 9999").show()

+-----------+----+-------+----------+-------+----------+-----------------+--------+
|employee_id| age|   city|department| salary| join_date|performance_score| courses|
+-----------+----+-------+----------+-------+----------+-----------------+--------+
|     9999.0|23.0|Phoenix|        HR|85183.0|2022-05-21|              1.8|    NULL|
|     9999.0|28.0|Chicago|        HR|60000.0|2023-09-01|              3.8|SEAS8515|
+-----------+----+-------+----------+-------+----------+-----------------+--------+



### 3(b) Update Scores

In [ ]:
from delta.tables import DeltaTable

delta_tbl = DeltaTable.forPath(spark, deltaPath)

delta_tbl.update(
    condition=col("city") == "Phoenix",
    set={"performance_score": col("performance_score") + 0.35}
)

## 4. Querying and Insights

### 4(a) Total Records

In [ ]:
# Register Delta table as a temporary view
df = spark.read.format("delta").load(deltaPath)
df.createOrReplaceTempView("employees")

spark.sql("SELECT COUNT(*) AS total_records FROM employees;").show()

+-------------+
|total_records|
+-------------+
|        10001|
+-------------+



### 4(b) Unique Cities

In [ ]:
spark.sql("SELECT COUNT(DISTINCT city) FROM employees").show()

+--------------------+
|count(DISTINCT city)|
+--------------------+
|                   5|
+--------------------+



## 5. Time Travel and Recovery

### 5(a) Previous Version

In [ ]:
deltaTable = DeltaTable.forPath(spark, deltaPath)

In [ ]:
deltaTable.history(4).select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

+-------+-----------------------+---------+----------------------------------------+
|version|timestamp              |operation|operationParameters                     |
+-------+-----------------------+---------+----------------------------------------+
|2      |2026-03-26 22:54:13.325|UPDATE   |{predicate -> ["(city#1229 = Phoenix)"]}|
|1      |2026-03-26 22:53:46.455|WRITE    |{mode -> Append, partitionBy -> []}     |
|0      |2026-03-26 22:51:20.554|WRITE    |{mode -> Overwrite, partitionBy -> []}  |
+-------+-----------------------+---------+----------------------------------------+



### 5(b) Explanation

Time travel is useful when incorrect data is written (e.g., a faulty update or delete). You can recover the prior state without requiring backups, enabling rollback, audit, or recovery in a simple and efficient way.

**Additional Use Cases:**

1. **Audit and Compliance:** Retrieve historical snapshots to verify what data looked like at a specific point in time — essential for financial, healthcare, and regulatory audits.

2. **Reprocessing Historical Data:** If a new model or rule needs to be tested on older data, time travel allows you to load that version without maintaining separate archives.

3. **Root Cause Analysis:** If downstream reports break due to a transformation error, you can roll back and compare versions to trace when and where the data pipeline issue occurred.

## 6. Advanced Analytics

### 6(a-c) Higher Order Functions

In [ ]:
spark.sql('''
SELECT department,
       size(filter(collect_list(performance_score), x -> x > 4.0)) AS high_scores
FROM employees
GROUP BY department
ORDER BY high_scores DESC
''').show()

+-----------+-----------+
| department|high_scores|
+-----------+-----------+
|  Marketing|        501|
|         HR|        496|
|    Finance|        491|
|Engineering|        475|
|      Sales|        468|
|       NULL|         60|
+-----------+-----------+



## 7. Static Delta Write

### 7(a) Static Write

In [ ]:
# Load static CSV data
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(dataPath)
)

# Save as Delta
df.write.format("delta").mode("overwrite").save(deltaPath)

## 8. Streaming Ingestion

### 8(a-c) Streaming

In [ ]:
def random_checkpoint_dir():
    return f"/tmp/chkpt/{random.randint(0, 10000)}"

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random

departments = ["Engineering", "Research", "Product", "Marketing"]
cities = ["Raleigh", "Austin", "Chicago", "Denver"]

@udf(returnType=StringType())
def random_city():
    return str(random.choice(cities))

@udf(returnType=StringType())
def random_department():
    return str(random.choice(departments))

In [ ]:
# Define streaming DataFrame using rate source
empStreamDF = (
    spark.readStream
    .format("rate")
    .load()
    .withColumn("employee_id", (10000 + col("value")).cast("double"))
    .withColumn("age", (rand() * 44 + 21).cast("double"))
    .withColumn("city", random_city())
    .withColumn("department", random_department())
    .withColumn("salary", (rand() * 110000 + 40000).cast("double"))
    .withColumn("join_date", current_date())
    .withColumn("performance_score", (rand() * 4 + 1).cast("double"))
    .select("employee_id", "age", "city", "department", "salary", "join_date", "performance_score")
)

# Start streaming write to Delta
query = (
    empStreamDF.writeStream
    .format("delta")
    .option("checkpointLocation", random_checkpoint_dir())
    .trigger(processingTime="10 seconds")
    .outputMode("append")
    .start(deltaPath)
)

## 9. Monitor Stream

### 9(a-b)

In [ ]:
spark.streams.active

## 10. Streaming Analytics

## 10 (a-b)

In [ ]:
# Register as temp table
spark.read.format("delta").load(deltaPath).createOrReplaceTempView("employees")

# Average performance by department
spark.sql("""
SELECT department, ROUND(AVG(performance_score), 2) AS avg_score
FROM employees
GROUP BY department
""").show()

# Employee count by department
spark.sql("""
SELECT department, COUNT(*) AS emp_count
FROM employees
GROUP BY department
""").show()

+-----------+---------+
| department|avg_score|
+-----------+---------+
|      Sales|     2.97|
|Engineering|     2.98|
|         HR|     3.03|
|       NULL|     3.03|
|    Finance|     3.01|
|   Research|     3.22|
|  Marketing|     2.97|
|    Product|     2.87|
+-----------+---------+

+-----------+---------+
| department|emp_count|
+-----------+---------+
|      Sales|     1976|
|Engineering|     2028|
|         HR|     1943|
|       NULL|      246|
|    Finance|     1920|
|   Research|       62|
|  Marketing|     1997|
|    Product|       47|
+-----------+---------+



## 11. Stop and Clean Up

### 11(a-b)

In [ ]:
for s in spark.streams.active:
    s.stop()

In [ ]:
spark.streams.active

[]